In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

all_sheets = pd.read_excel('/content/drive/MyDrive/Cadetx /ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']

print(sessions_df[['start_timestamp', 'total_cost', 'price_per_kwh']].head())

      start_timestamp  total_cost  price_per_kwh
0 2022-01-01 08:30:00       26.74           0.47
1 2022-01-01 09:20:00       18.72           0.46
2 2022-01-01 09:00:00       14.52           0.47
3 2022-01-01 23:20:00       12.78           0.45
4 2022-01-01 16:10:00       17.75           0.56


In [3]:
sessions_df['date'] = sessions_df['start_timestamp'].dt.date
daily_revenue = sessions_df.groupby('date')['total_cost'].sum().reset_index()
daily_revenue.columns = ['date', 'daily_revenue']

print(daily_revenue.head())
print(daily_revenue.shape)

         date  daily_revenue
0  2022-01-01        2522.63
1  2022-01-02        1982.62
2  2022-01-03        3203.87
3  2022-01-04        2873.93
4  2022-01-05        3116.46
(1096, 2)


In [4]:
print(sessions_df['price_per_kwh'].describe())

count    294024.000000
mean          0.592319
std           0.063600
min           0.450000
25%           0.540000
50%           0.590000
75%           0.640000
max           0.800000
Name: price_per_kwh, dtype: float64


In [5]:
sessions_df['price_bucket'] = sessions_df['price_per_kwh'].round(2)

price_elasticity = sessions_df.groupby('price_bucket').agg(
    session_count=('session_id', 'count'),
    avg_energy_kwh=('energy_kwh', 'mean')
).reset_index()

print(price_elasticity.sort_values('price_bucket'))

    price_bucket  session_count  avg_energy_kwh
0           0.45           1544       49.723446
1           0.46           3127       49.389415
2           0.47           3188       49.631556
3           0.48           3278       49.541977
4           0.49           3563       49.429498
5           0.50           8147       47.863999
6           0.51          12537       47.298492
7           0.52          12465       47.240674
8           0.53          12746       47.325498
9           0.54          13463       47.523628
10          0.55          14285       46.939608
11          0.56          15558       46.597609
12          0.57          15888       46.856898
13          0.58          16442       46.481182
14          0.59          16283       46.654370
15          0.60          16308       46.666268
16          0.61          16291       46.710294
17          0.62          16457       46.519335
18          0.63          16554       46.560529
19          0.64          16563       46

In [6]:
from prophet import Prophet

prophet_revenue = daily_revenue.rename(columns={'date': 'ds', 'daily_revenue': 'y'})
prophet_revenue['ds'] = pd.to_datetime(prophet_revenue['ds'])

print(prophet_revenue.head())

          ds        y
0 2022-01-01  2522.63
1 2022-01-02  1982.62
2 2022-01-03  3203.87
3 2022-01-04  2873.93
4 2022-01-05  3116.46


In [7]:
model_revenue = Prophet()
model_revenue.fit(prophet_revenue)

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [8]:
future_revenue = model_revenue.make_future_dataframe(periods=180)  # roughly 6 months ahead
forecast_revenue = model_revenue.predict(future_revenue)

print(forecast_revenue[['ds', 'yhat']].tail())

             ds          yhat
1271 2025-06-25  17615.977568
1272 2025-06-26  17599.112221
1273 2025-06-27  17637.081049
1274 2025-06-28  15464.455195
1275 2025-06-29  15478.212864


In [10]:
current_avg_price = sessions_df['price_per_kwh'].mean()
current_total_revenue = sessions_df['total_cost'].sum()

In [11]:
print(f"Current average price: ${current_avg_price:.2f}/kWh")
print(f"Current total revenue: ${current_total_revenue:,.2f}")

Current average price: $0.59/kWh
Current total revenue: $8,171,100.12


In [13]:
# Scenario 1: Price +5%
revenue_plus_5pct = current_total_revenue * 1.05
print(f"\nScenario: Price +5% → Revenue: ${revenue_plus_5pct:,.2f}")



Scenario: Price +5% → Revenue: $8,579,655.13


In [14]:
# Scenario 2: Price -10%
revenue_minus_10pct = current_total_revenue * 0.90
print(f"Scenario: Price -10% → Revenue: ${revenue_minus_10pct:,.2f}")

Scenario: Price -10% → Revenue: $7,353,990.11


In [15]:
daily_revenue.to_csv('/content/drive/MyDrive/Cadetx /daily_revenue.csv', index=False)
price_elasticity.to_csv('/content/drive/MyDrive/Cadetx /price_elasticity.csv', index=False)

print("Saved!")

Saved!
